# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook demonstrates how to explore and process a dataset described by a Croissant schema using the `mlcroissant` library and referencing all entities via their `@id`s.

### Dataset Source
Croissant schema URL: https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display dataset main information
meta = dataset.metadata
print(f"{meta.name}\n\n{meta.description}")

## 2. Data Overview
Review available record sets, their IDs, and fields in the dataset.

In [ ]:
# List all record sets and their @id
print("Available record sets (by @id):")
if hasattr(dataset.metadata, 'recordSet') and dataset.metadata.recordSet:
    if isinstance(dataset.metadata.recordSet, list):
        record_sets = dataset.metadata.recordSet
    else:
        record_sets = [dataset.metadata.recordSet]
else:
    # If not present in metadata, try dataset.record_sets()
    record_sets = [r['@id'] for r in dataset.record_sets()]
    
print(record_sets)

# Show fields for each record set by @id
for record_set_id in record_sets:
    record_set_desc = dataset.record_set(record_set_id)
    print(f"\nFields in record set {record_set_id}: ")
    if 'fields' in record_set_desc:
        for field in record_set_desc['fields']:
            # Each field is a dict with '@id' and possibly 'name'
            if isinstance(field, dict):
                print(f"  - {field.get('@id', 'unknown')} ({field.get('name', '')})")
            else:
                print(f"  - {field}")
    else:
        print("  No fields found in this record set.")

## 3. Data Extraction
Load data from each record set into pandas DataFrames for analysis. All extraction references record set and field `@id`s.

In [ ]:
# Dynamically list all record sets by @id
record_set_ids = [r['@id'] for r in dataset.record_sets()] if dataset.record_sets() else []
if not record_set_ids:
    print("No record sets found.")
else:
    dataframes = dict()
    # Load each record set into a DataFrame
    for rs_id in record_set_ids:
        try:
            records = list(dataset.records(record_set=rs_id))
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} rows for record set {rs_id}.")
        except Exception as e:
            print(f"Could not load record set {rs_id}: {str(e)}")

    # Show columns for one record set (first)
    if dataframes:
        first_rs = record_set_ids[0]
        print(f"\nColumns for record set {first_rs}:\n{dataframes[first_rs].columns.tolist()}")
        display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
We demonstrate numeric filtering, normalization, and grouping using sample field `@id`s. Please edit field `@id`s below to match the fields from your dataset overview as appropriate.

In [ ]:
# Example EDA: Filter, normalize, and group numeric fields

# Choose a record set for EDA (update ID as needed)
eda_rs_id = record_set_ids[0] if record_set_ids else None
if eda_rs_id and eda_rs_id in dataframes:
    df = dataframes[eda_rs_id]
    print(f"Fields/columns for record set {eda_rs_id}:\n{df.columns.tolist()}")
    # Choose a numeric field by examining the columns
    # Replace <numeric_field_id> and <group_field_id> with actual field @id strings from the output above
    example_numeric_field_id = df.select_dtypes(include='number').columns[0] if not df.select_dtypes(include='number').empty else None
    if example_numeric_field_id:
        threshold = df[example_numeric_field_id].mean()  # Example threshold
        filtered_df = df[df[example_numeric_field_id] > threshold].copy()
        print(f"\nFiltered records where {example_numeric_field_id} > {threshold:.2f} (mean):\n", filtered_df.head())

        norm_col = f"{example_numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[example_numeric_field_id] - filtered_df[example_numeric_field_id].mean()) / filtered_df[example_numeric_field_id].std()
        print(f"\nNormalized column {example_numeric_field_id} in filtered records:")
        display(filtered_df[[example_numeric_field_id, norm_col]].head())

        # Try to group by a likely categorical field
        possible_group_fields = df.select_dtypes(include=['object', 'category']).columns
        if len(possible_group_fields) > 0:
            group_field_id = possible_group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id)[example_numeric_field_id].mean().to_frame()
            print(f"\nGrouped mean {example_numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No categorical fields available for grouping.")
    else:
        print("No numeric fields detected in this record set. Please adjust field selection for EDA.")
else:
    print("No data available in dataframes for EDA.")

## 5. Visualization
We can plot distributions or relationships for chosen numeric fields. Replace field `@id`s as necessary based on your dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Simple visualization example
if eda_rs_id and eda_rs_id in dataframes and example_numeric_field_id:
    fig, ax = plt.subplots(figsize=(7,4))
    sns.histplot(dataframes[eda_rs_id][example_numeric_field_id].dropna(), kde=True, ax=ax)
    ax.set_title(f'Distribution of {example_numeric_field_id}')
    plt.xlabel(example_numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If possible, pair a numeric field and a category for boxplot
    if len(possible_group_fields) > 0:
        group_field_id = possible_group_fields[0]
        plt.figure(figsize=(10,4))
        sns.boxplot(x=dataframes[eda_rs_id][group_field_id], y=dataframes[eda_rs_id][example_numeric_field_id])
        plt.title(f'{example_numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable data for visualization.")

## 6. Conclusion
In this notebook we demonstrated how to discover, extract, and analyze record sets and fields from a Croissant-annotated dataset using `mlcroissant`, referencing all components by their `@id`. You can extend and adapt the analysis and visualization steps based on the schema and field details revealed in the overview.